# PPO Self-Play: EdgeTransformerNet Actor + TransformerCritic

Fine-tunes the imitation-pretrained `EdgeTransformerNet` via PPO self-play.

## Required Kaggle inputs

| Dataset slug | Contents | Required? |
|---|---|---|
| `orbitswar-src` | Full project source | Always |
| `orbitswar-imitation` | `best_model.pt` from imitation training | Always |
| `orbitswar-rl` | `transformer_last.pt` from previous session | Resume only |

## Iterative session strategy

Run 20 rounds per session (~4.3h on T4), analyse, then resume:

| Session | Attach RL? | Rounds | What to look for |
|---|---|---|---|
| 1 (fresh) | No | 1–20 | Critic loss falling — critic calibrating |
| 2 (resume) | Yes | 21–40 | WR trending up after round 20 |
| 3+ (resume) | Yes | 41–60, 61–80... | Continued WR improvement |

## Fixes applied vs previous broken run

| Problem | Fix |
|---|---|
| Adaptive entropy hitting ceiling → loss just tracking entropy | Removed — replaced with fixed `ENT_COEF=0.01` |
| Terminal reward replacing last shaped reward (`=` instead of `+=`) | Fixed — terminal now added on top |
| 8 games/round → noisy WR signal | Raised to 16 games/round |
| Old broken checkpoint resuming with bad critic | Fresh start — delete RL dataset for session 1 |

## Output
- `rl/transformer_last.pt` — checkpoint after every round (download + upload as orbitswar-rl to resume)
- `rl/transformer_best.pt` — best win-rate checkpoint
- `ppo_training.log` — full training log

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 1 — Install orbit_wars environment
# ─────────────────────────────────────────────────────────
%pip install --upgrade "kaggle-environments>=1.28.0"

# Verify orbit_wars is available
from kaggle_environments import make
env = make('orbit_wars', configuration={'episodeSteps': 5})
print('orbit_wars OK')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2b — Logging setup
# Writes to 3 places simultaneously:
#   1. ppo_training.log  — download from Output tab at end
#   2. stderr            — shows LIVE in Kaggle Logs tab during run
#   3. Jupyter stdout    — shows in cell output when interactive
# ─────────────────────────────────────────────────────────
import sys, time
from pathlib import Path

LOG_PATH = Path('/kaggle/working/ppo_training.log')

class _TeeLogger:
    def __init__(self, orig, path):
        self._orig = orig
        self._file = open(path, 'w', buffering=1, encoding='utf-8')
        ts = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
        self._file.write('# OrbitsWar — PPO Training Log\n')
        self._file.write(f'# Started : {ts}\n\n')
        self._file.flush()

    def write(self, msg):
        # 1. Jupyter cell output
        self._orig.write(msg)
        try: self._orig.flush()
        except: pass
        # 2. stderr → live in Kaggle Logs tab during committed runs
        try:
            sys.__stderr__.write(msg)
            sys.__stderr__.flush()
        except: pass
        # 3. log file
        try:
            self._file.write(msg)
            self._file.flush()
        except: pass

    def flush(self):
        try: self._orig.flush()
        except: pass
        try: sys.__stderr__.flush()
        except: pass
        try: self._file.flush()
        except: pass

    def isatty(self):
        return getattr(self._orig, 'isatty', lambda: False)()

    def close(self):
        sys.stdout = self._orig
        try: self._file.flush(); self._file.close()
        except: pass

_tee = _TeeLogger(sys.stdout, LOG_PATH)
sys.stdout = _tee

print(f'Logging to : {LOG_PATH}')
print('(stdout → ppo_training.log + Kaggle Logs tab live)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 2 — Check GPU
# ─────────────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 3 — Dataset paths
# ─────────────────────────────────────────────────────────
from pathlib import Path

SRC_DATASET       = 'orbitswar-src'        # source code (required)
IMITATION_DATASET = 'orbitswar-imitation'  # best_model.pt (required)

SRC_DIR  = Path(f'/kaggle/input/{SRC_DATASET}')
IMIT_DIR = Path(f'/kaggle/input/{IMITATION_DATASET}')

print(f'SRC  exists: {SRC_DIR.exists()}   ({SRC_DIR})')
print(f'IMIT exists: {IMIT_DIR.exists()}  ({IMIT_DIR})')

# RL dataset is only needed to RESUME a previous session.
# On a fresh start, leave it unattached — RL_DIR will be None.
_rl_candidate = Path('/kaggle/input/orbitswar-rl')
RL_DIR = _rl_candidate if _rl_candidate.exists() else None

if RL_DIR:
    print(f'RL   found : {RL_DIR}  ← will resume from previous session')
else:
    print('RL   not attached — fresh start (no prior PPO checkpoints needed)')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 4 — Copy source tree to /kaggle/working/
# ─────────────────────────────────────────────────────────
import os, sys, shutil

WORK = Path('/kaggle/working')

# Copy everything from the source dataset
for item in SRC_DIR.iterdir():
    dst = WORK / item.name
    if item.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(item, dst)
    else:
        shutil.copy2(item, dst)
print('Source tree copied.')

# ── Copy imitation checkpoint ─────────────────────────────
imit_ckpt_dst = WORK / 'imitation' / 'checkpoints'
imit_ckpt_dst.mkdir(parents=True, exist_ok=True)
imit_src = IMIT_DIR / 'best_model.pt'
if imit_src.exists():
    shutil.copy2(imit_src, imit_ckpt_dst / 'best_model.pt')
    print(f'Imitation checkpoint copied -> {imit_ckpt_dst}/best_model.pt')
else:
    print('WARNING: best_model.pt not found — actor will start from random init.')

# ── Copy transformer checkpoints (resume only if RL_DIR attached) ────────────
rl_dst = WORK / 'rl'
rl_dst.mkdir(parents=True, exist_ok=True)
if RL_DIR is not None:
    resumed = []
    for name in ['transformer_last.pt', 'transformer_best.pt']:
        src = RL_DIR / name
        if src.exists():
            shutil.copy2(src, rl_dst / name)
            resumed.append(name)
    if resumed:
        print(f'PPO checkpoints copied -> rl/: {resumed}')
        print('  trainer will resume automatically from transformer_last.pt')
    else:
        print('RL dataset attached but no .pt files found — starting fresh.')
else:
    print('Fresh start — no prior PPO checkpoints.')

# ── Ensure enemy agents directory exists ──────────────────
enemy_dir = WORK / 'enemy agents'
if not enemy_dir.exists():
    enemy_dir.mkdir(parents=True)
std_bots = list(enemy_dir.glob('*/main.py'))
my_bots  = list((enemy_dir / 'my').glob('*.py')) if (enemy_dir / 'my').exists() else []
if not std_bots and not my_bots:
    self_bot_dir = enemy_dir / 'self_main'
    self_bot_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(WORK / 'main.py', self_bot_dir / 'main.py')
    print('No enemies found — added main.py as self-play opponent.')
else:
    labels = [p.parent.name for p in std_bots] + [p.stem for p in my_bots]
    print(f'Enemies: {labels}')

# ── Python path ───────────────────────────────────────────
for p in [str(WORK), str(WORK/'rl'), str(WORK/'imitation'), str(WORK/'classifier')]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('\nWorking directory contents:')
for p in sorted(WORK.iterdir()):
    print(f'  {p.name}/' if p.is_dir() else f'  {p.name}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 5 — Verify imports
# ─────────────────────────────────────────────────────────
os.chdir(WORK)   # make relative imports work from project root

from imitation.model    import EdgeTransformerNet, load_model as load_actor
from imitation.features import build_all_candidate_edges

# Count actor parameters
actor_test = EdgeTransformerNet()
n_params = sum(p.numel() for p in actor_test.parameters())
print(f'EdgeTransformerNet params: {n_params:,}')
del actor_test

# Check imitation checkpoint
imit_ckpt = WORK / 'imitation' / 'checkpoints' / 'best_model.pt'
print(f'Imitation checkpoint exists: {imit_ckpt.exists()}')
if imit_ckpt.exists():
    import torch
    ckpt = torch.load(imit_ckpt, map_location='cpu', weights_only=False)
    print(f'  Trained epoch: {ckpt.get("epoch", "?")}  '
          f'val_loss: {ckpt.get("val_loss", float("nan")):.4f}')

print('\nAll imports OK.')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 6 — Training hyperparameters
# ─────────────────────────────────────────────────────────
import rl.transformer_trainer as TT

# Iterative session strategy — run 20 rounds per session:
#   Session 1 (fresh):  rounds  1-20  → all critic warmup (actor frozen)
#                       → download transformer_last.pt, upload as orbitswar-rl
#   Session 2 (resume): rounds 21-40  → first real PPO, check WR trend
#                       → if WR trending up, continue; else diagnose
#   Session 3+ (resume):rounds 41-60, 61-80, etc.
#
# ~13 min/round on T4 → 20 rounds ≈ 4.3h per session
N_ROUNDS        = 20
GAMES_PER_ROUND = 16

LR_ACTOR  = 5e-5
LR_CRITIC = 3e-4
CLIP_EPS   = 0.2
PPO_EPOCHS = 4

print(f'Rounds          : {N_ROUNDS}  (~{N_ROUNDS * 13 / 60:.1f}h on T4)')
print(f'Games / round   : {GAMES_PER_ROUND}  (±{100/GAMES_PER_ROUND:.1f}% WR resolution)')
print(f'ENT_COEF        : {TT.ENT_COEF} (fixed)')
print(f'Critic warm-up  : {TT.CRITIC_WARMUP_ROUNDS} rounds — actor frozen, critic calibrates first')
print()
print('Iterative plan:')
print('  Session 1 (fresh)   → rounds  1-20  critic warmup only')
print('  Session 2 (resume)  → rounds 21-40  first PPO signal')
print('  Session 3+ (resume) → rounds 41+    continue if trending up')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 7 — Patch trainer hyperparameters, then launch
# ─────────────────────────────────────────────────────────
import re, matplotlib.pyplot as plt
import rl.transformer_trainer as TT

TT.N_ROUNDS        = N_ROUNDS
TT.GAMES_PER_ROUND = GAMES_PER_ROUND
TT.LR_ACTOR        = LR_ACTOR
TT.LR_CRITIC       = LR_CRITIC
TT.CLIP_EPS        = CLIP_EPS
TT.PPO_EPOCHS      = PPO_EPOCHS

TT.SAVE_PATH = WORK / 'rl' / 'transformer_last.pt'
TT.BEST_PATH = WORK / 'rl' / 'transformer_best.pt'

print(f'Checkpoints -> Last : {TT.SAVE_PATH}')
print(f'               Best : {TT.BEST_PATH}')
print(f'Critic warm-up      : {TT.CRITIC_WARMUP_ROUNDS} rounds (actor frozen)')
print(f'ENT_COEF            : {TT.ENT_COEF} (fixed)')
print(f'Log file            : /kaggle/working/ppo_training.log')
print()

# ── Helper: parse log and plot curves ────────────────────
def _plot_curves():
    log_path = Path('/kaggle/working/ppo_training.log')
    # log line: -> WR=37.5%  loss=-0.1565  ent=4.2457  [warmup/tag]
    pattern = re.compile(
        r'WR=(?P<wr>[\d.]+)%\s+loss=(?P<loss>[-\d.]+)\s+ent=(?P<ent>[\d.]+)'
    )
    rounds, wrs, losses, entropies, best_wrs = [], [], [], [], []
    best_running = 0.0

    if log_path.exists():
        for line in log_path.read_text(encoding='utf-8').splitlines():
            m = pattern.search(line)
            if m:
                wr = float(m.group('wr'))
                rounds.append(len(rounds) + 1)
                wrs.append(wr)
                losses.append(float(m.group('loss')))
                entropies.append(float(m.group('ent')))
                best_running = max(best_running, wr)
                best_wrs.append(best_running)

    if not rounds:
        print('No training rounds in log yet.')
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(rounds, wrs,      alpha=0.35, color='steelblue', label='WR (round)')
    axes[0].plot(rounds, best_wrs, color='steelblue', linewidth=2, label='Best WR')
    axes[0].axhline(50, color='g', linestyle='--', alpha=0.5, label='50% target')
    axes[0].set_title('Win Rate %'); axes[0].set_xlabel('Round')
    axes[0].set_ylabel('%'); axes[0].legend(); axes[0].set_ylim(0, 100)

    axes[1].plot(rounds, losses, color='crimson', label='PPO loss')
    axes[1].set_title('PPO Loss'); axes[1].set_xlabel('Round'); axes[1].legend()

    axes[2].plot(rounds, entropies, color='orange', label='Entropy')
    axes[2].axhline(1.5, color='g', linestyle=':', alpha=0.5, label='Good range ~1.5-3.5')
    axes[2].axhline(3.5, color='g', linestyle=':', alpha=0.5)
    axes[2].set_title('Entropy (nats)'); axes[2].set_xlabel('Round')
    axes[2].set_ylabel('nats'); axes[2].legend()

    best_idx = best_wrs.index(max(best_wrs))
    plt.suptitle(
        f'PPO Training — EdgeTransformerNet  '
        f'(best WR={max(best_wrs):.1f}% @ round {rounds[best_idx]},  '
        f'{len(rounds)} rounds total)',
        fontsize=11)
    plt.tight_layout()

    curve_path = WORK / 'rl' / 'ppo_training_curves.png'
    plt.savefig(curve_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Curve saved : {curve_path}')
    print(f'Parsed      : {len(rounds)} rounds  |  '
          f'Best WR={max(best_wrs):.1f}% @ round {rounds[best_idx]}  |  '
          f'Final ent={entropies[-1]:.3f}')

# ── Run training — plot on finish or interrupt ────────────
actor = critic = None
try:
    actor, critic = TT.train(
        n_rounds        = N_ROUNDS,
        games_per_round = GAMES_PER_ROUND,
        actor_path      = str(WORK / 'imitation' / 'checkpoints' / 'best_model.pt'),
        device          = DEVICE,
    )
except KeyboardInterrupt:
    print('\nTraining interrupted.')
finally:
    print('\n── Training curves ──────────────────────────────────')
    _plot_curves()

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8 — Quick sanity evaluation (5 games vs self)
# ─────────────────────────────────────────────────────────
from rl.transformer_trainer import play_game_collect

if actor is None:
    print('Training returned None — skipping eval.')
else:
    wins = losses = draws = 0
    from rl.self_play_trainer import load_agent
    main_fn, _ = load_agent(str(WORK / 'main.py'))
    fns = [None, main_fn, main_fn, main_fn]  # player 0 = transformer

    print('Quick eval (5 games vs base main.py)...')
    for g in range(5):
        buf, outcome = play_game_collect(actor, critic, fns,
                                         seed=9000 + g, device=DEVICE)
        if   outcome == 'WIN':  wins   += 1
        elif outcome == 'LOSS': losses += 1
        else:                   draws  += 1
        print(f'  Game {g+1}: {outcome}  (W:{wins} L:{losses} D:{draws})')

    print(f'\nWin rate: {wins}/5 = {wins/5*100:.0f}%')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 8b — Plot PPO training curves (parsed from log)
#            Run this cell any time to see current progress
# ─────────────────────────────────────────────────────────
import re
import matplotlib.pyplot as plt

log_path = Path('/kaggle/working/ppo_training.log')
if not log_path.exists():
    print('No log found — run training first.')
else:
    # log line: -> WR=37.5%  loss=-0.1565  ent=4.2457  [warmup/tag]
    pattern = re.compile(
        r'WR=(?P<wr>[\d.]+)%\s+loss=(?P<loss>[-\d.]+)\s+ent=(?P<ent>[\d.]+)'
    )

    rounds, wrs, losses, entropies = [], [], [], []
    best_wr_running = 0.0
    best_wrs = []

    for line in log_path.read_text(encoding='utf-8').splitlines():
        m = pattern.search(line)
        if m:
            wr = float(m.group('wr'))
            rounds.append(len(rounds) + 1)
            wrs.append(wr)
            losses.append(float(m.group('loss')))
            entropies.append(float(m.group('ent')))
            best_wr_running = max(best_wr_running, wr)
            best_wrs.append(best_wr_running)

    if not rounds:
        print('No round entries found in log yet.')
    else:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        # Win rate
        axes[0].plot(rounds, wrs,      alpha=0.35, color='steelblue', label='WR (round)')
        axes[0].plot(rounds, best_wrs, color='steelblue', linewidth=2, label='Best WR')
        axes[0].axhline(50, color='g', linestyle='--', alpha=0.5, label='50% target')
        axes[0].set_title('Win Rate %')
        axes[0].set_xlabel('Round'); axes[0].set_ylabel('%')
        axes[0].legend(); axes[0].set_ylim(0, 100)

        # PPO loss
        axes[1].plot(rounds, losses, color='crimson', label='PPO loss')
        axes[1].set_title('PPO Loss'); axes[1].set_xlabel('Round'); axes[1].legend()

        # Entropy (no alpha — fixed ENT_COEF now)
        axes[2].plot(rounds, entropies, color='orange', label='Entropy')
        axes[2].axhline(1.5, color='g', linestyle=':', alpha=0.5, label='Good range ~1.5-3.5')
        axes[2].axhline(3.5, color='g', linestyle=':', alpha=0.5)
        axes[2].set_title('Entropy (nats)')
        axes[2].set_xlabel('Round'); axes[2].set_ylabel('nats')
        axes[2].legend()

        best_idx = best_wrs.index(max(best_wrs))
        plt.suptitle(
            f'PPO Training — EdgeTransformerNet  '
            f'(best WR={max(best_wrs):.1f}% @ round {rounds[best_idx]})',
            fontsize=11)
        plt.tight_layout()

        out = WORK / 'rl' / 'ppo_training_curves.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
        plt.show()

        print(f'Parsed {len(rounds)} rounds from log.')
        print(f'Best WR : {max(best_wrs):.1f}%  @ round {rounds[best_idx]}')
        print(f'Final   : WR={wrs[-1]:.1f}%  loss={losses[-1]:.4f}  ent={entropies[-1]:.4f}')
        print(f'Saved   : {out}')

In [ ]:
# ─────────────────────────────────────────────────────────
# CELL 9 — List output files (download from Output tab)
# ─────────────────────────────────────────────────────────
import os

print('Output checkpoints:')
for pt in sorted((WORK / 'rl').glob('transformer*.pt')):
    size_mb = pt.stat().st_size / 1e6
    print(f'  {pt.name}  ({size_mb:.1f} MB)')

print()
print('To use in submission:')
print('  1. Download transformer_best.pt')
print('  2. Place in imitation/checkpoints/best_model.pt')
print('  3. Your agent_nn.py will pick it up automatically')

# ── Flush & close log ─────────────────────────────────────
try:
    import time as _t
    print(f'\n# Log ended : {_t.strftime("%Y-%m-%d %H:%M:%S UTC", _t.gmtime())}')
    _tee.flush()
    _log_sz = LOG_PATH.stat().st_size
    _tee.close()                       # restores sys.stdout to original
    print(f'\nLog saved   : {LOG_PATH}  ({_log_sz / 1024:.1f} KB)')
    print('Download ppo_training.log from the Output tab.')
except Exception as _log_err:
    print(f'Log close error: {_log_err}')